# Site-Level Threshold Optimization

This notebook finds a reasonable **site-level** decision threshold for the final 14-3-3 site model.

It uses:

- `data/final_dataset_with_all_new.pkl`
- the final selected feature set from the app/backend
- the trained site-level classifier in `model/1433model_20260223.pkl`

The threshold is optimized on a validation split built from the `training` portion of the dataset, while keeping
the same protein group (`augmentated from uniprot ID`) in only one split.


In [1]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

from features import FEATURE_COLUMNS


/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and model


In [2]:
df = pd.read_pickle("data/final_dataset_with_all_new.pkl")
model = joblib.load("model/1433model_20260223.pkl")

group_col = "augmentated from uniprot ID"
target_col = "label"
usage_col = "usage"

print(df.shape)
print(df[usage_col].value_counts())
print(df[target_col].value_counts())


(2793, 4454)
usage
training            2266
independent test     527
Name: count, dtype: int64
label
0    1463
1    1330
Name: count, dtype: int64


## Keep only rows needed for threshold selection

We use the rows marked as `training`, then split by protein group into train/validation groups.


In [3]:
train_df = df[df[usage_col] == "training"].copy()
test_df = df[df[usage_col] == "independent test"].copy()

required_columns = FEATURE_COLUMNS + [target_col, group_col]
train_df = train_df[required_columns].copy()
test_df = test_df[FEATURE_COLUMNS + [target_col]].copy()

print(train_df.shape)
print(test_df.shape)


(2266, 46)
(527, 45)


## Build a validation split by protein group

This follows the same spirit as your modeling notebook: rows from the same protein group should not appear in both
training and validation.


In [4]:
group_label_df = train_df.groupby(group_col)[target_col].max().reset_index()

train_groups, valid_groups = train_test_split(
    group_label_df[group_col],
    test_size=0.2,
    random_state=42,
    stratify=group_label_df[target_col],
)

train_split_df = train_df[train_df[group_col].isin(train_groups)].copy()
valid_split_df = train_df[train_df[group_col].isin(valid_groups)].copy()

print("train rows:", train_split_df.shape)
print("valid rows:", valid_split_df.shape)
print("train label counts:\n", train_split_df[target_col].value_counts())
print("valid label counts:\n", valid_split_df[target_col].value_counts())


train rows: (1826, 46)
valid rows: (440, 46)
train label counts:
 label
0    968
1    858
Name: count, dtype: int64
valid label counts:
 label
0    231
1    209
Name: count, dtype: int64


## Get site-level probabilities from the trained model


In [5]:
X_valid = valid_split_df[FEATURE_COLUMNS]
y_valid = valid_split_df[target_col]

valid_proba = model.predict_proba(X_valid)[:, 1]
valid_pred_default = model.predict(X_valid)

print("Default threshold metrics from model.predict():")
print("Accuracy:", round(accuracy_score(y_valid, valid_pred_default), 4))
print("MCC:", round(matthews_corrcoef(y_valid, valid_pred_default), 4))
print("F1:", round(f1_score(y_valid, valid_pred_default), 4))


Default threshold metrics from model.predict():
Accuracy: 0.9364
MCC: 0.8736
F1: 0.9346


## Sweep thresholds

We evaluate a grid of thresholds and select the one with the best MCC on the validation set.


In [6]:
def evaluate_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }


threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
threshold_results = pd.DataFrame(
    [evaluate_threshold(y_valid, valid_proba, threshold) for threshold in threshold_grid]
)

threshold_results = threshold_results.sort_values(
    ["mcc", "f1", "accuracy", "threshold"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

threshold_results.head(10)


,threshold,accuracy,precision,recall,f1,mcc
0,0.58,0.943182,0.946602,0.933014,0.939759,0.886084
1,0.61,0.940909,0.955224,0.918660,0.936585,0.881894
2,0.60,0.940909,0.950739,0.923445,0.936893,0.881690
3,0.59,0.940909,0.946341,0.928230,0.937198,0.881561
4,0.62,0.938636,0.955000,0.913876,0.933985,0.877454
5,0.63,0.938636,0.955000,0.913876,0.933985,0.877454
6,0.36,0.936364,0.891775,0.985646,0.936364,0.877421
7,0.53,0.938636,0.925234,0.947368,0.936170,0.877332
8,0.55,0.938636,0.933333,0.937799,0.935561,0.877002
9,0.38,0.936364,0.895197,0.980861,0.936073,0.876602


## Best threshold


In [19]:
best_threshold = float(threshold_results.iloc[0]["threshold"])
best_threshold


0.58

In [20]:
best_valid_pred = (valid_proba >= best_threshold).astype(int)

print("Best validation threshold:", best_threshold)
print("Validation Accuracy:", round(accuracy_score(y_valid, best_valid_pred), 4))
print("Validation Precision:", round(precision_score(y_valid, best_valid_pred, zero_division=0), 4))
print("Validation Recall:", round(recall_score(y_valid, best_valid_pred, zero_division=0), 4))
print("Validation F1:", round(f1_score(y_valid, best_valid_pred, zero_division=0), 4))
print("Validation MCC:", round(matthews_corrcoef(y_valid, best_valid_pred), 4))
print("Validation Confusion Matrix:")
print(confusion_matrix(y_valid, best_valid_pred))
print(classification_report(y_valid, best_valid_pred, zero_division=0))


Best validation threshold: 0.58
Validation Accuracy: 0.9432
Validation Precision: 0.9466
Validation Recall: 0.933
Validation F1: 0.9398
Validation MCC: 0.8861
Validation Confusion Matrix:
[[220  11]
 [ 14 195]]
              precision    recall  f1-score   support

           0       0.94      0.95      0.95       231
           1       0.95      0.93      0.94       209

    accuracy                           0.94       440
   macro avg       0.94      0.94      0.94       440
weighted avg       0.94      0.94      0.94       440



## Evaluate this threshold on the independent test set


In [21]:
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[target_col]

test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

print("Independent test threshold:", best_threshold)
print("Test Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Test Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Test Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("Test F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))
print("Test MCC:", round(matthews_corrcoef(y_test, test_pred), 4))
print("Test Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, zero_division=0))


Independent test threshold: 0.58
Test Accuracy: 0.8273
Test Precision: 0.9526
Test Recall: 0.6882
Test F1: 0.7991
Test MCC: 0.6812
Test Confusion Matrix:
[[255   9]
 [ 82 181]]
              precision    recall  f1-score   support

           0       0.76      0.97      0.85       264
           1       0.95      0.69      0.80       263

    accuracy                           0.83       527
   macro avg       0.85      0.83      0.82       527
weighted avg       0.85      0.83      0.82       527



## Save or reuse the selected threshold

You can manually take `best_threshold` and use it later in your app or notebook workflows.
